In [1]:
API_KEY = "cf64399ff0e7ee8a2270a76b7e0767156b22c49ad79e6aa6a38836a80ee41aa1"

In [2]:
import requests

headers = {"X-API-Key": API_KEY}

def fetch_locations(country_ids: dict) -> dict:
    """Takes {country_name: country_id} and returns {country_name: list_of_locations}."""
    results = {}
    for country, country_id in country_ids.items():
        all_locations = []
        page = 1
        while True:
            response = requests.get(
                "https://api.openaq.org/v3/locations",
                params={"countries_id": country_id, "limit": 1000, "page": page,},# "providers_id": 119},
                headers=headers
            )
            data = response.json()
            page_results = data["results"]
            if not page_results:
                break
            all_locations.extend(page_results)
            print(f"{country} page {page}: fetched {len(page_results)} | total: {len(all_locations)}")
            page += 1
        results[country] = all_locations
    return results


In [3]:
codes={"Chile": 3}#"Germany": 50, "France": 22, "Australia": 177, "Brazil": 45}


In [4]:
locations = fetch_locations(codes)

Chile page 1: fetched 185 | total: 185


In [5]:
locations

{'Chile': [{'id': 25,
   'name': "Parque O'Higgins",
   'locality': 'Santiago',
   'timezone': 'America/Santiago',
   'country': {'id': 3, 'code': 'CL', 'name': 'Chile'},
   'owner': {'id': 4, 'name': 'Unknown Governmental Organization'},
   'provider': {'id': 164, 'name': 'Chile - SINCA'},
   'isMobile': False,
   'isMonitor': True,
   'instruments': [{'id': 2, 'name': 'Government Monitor'}],
   'sensors': [{'id': 1045,
     'name': 'co µg/m³',
     'parameter': {'id': 4,
      'name': 'co',
      'units': 'µg/m³',
      'displayName': 'CO mass'}},
    {'id': 1046,
     'name': 'no2 µg/m³',
     'parameter': {'id': 5,
      'name': 'no2',
      'units': 'µg/m³',
      'displayName': 'NO₂ mass'}},
    {'id': 114,
     'name': 'o3 µg/m³',
     'parameter': {'id': 3,
      'name': 'o3',
      'units': 'µg/m³',
      'displayName': 'O₃ mass'}},
    {'id': 1047,
     'name': 'pm10 µg/m³',
     'parameter': {'id': 1,
      'name': 'pm10',
      'units': 'µg/m³',
      'displayName': 'PM10'}

In [6]:
def filter_monitors(locations: dict) -> dict:
    return {country: [loc for loc in locs if loc["isMonitor"]] for country, locs in locations.items()}

monitor_locations = filter_monitors(locations)
{country: len(locs) for country, locs in monitor_locations.items()}


{'Chile': 176}

In [7]:
location_ids = {country: [loc["id"] for loc in locs] for country, locs in monitor_locations.items()}
location_ids

{'Chile': [25,
  26,
  27,
  38,
  39,
  45,
  47,
  54,
  61,
  65,
  67,
  68,
  69,
  72,
  73,
  81,
  136,
  139,
  140,
  145,
  210,
  270,
  271,
  292,
  294,
  295,
  296,
  299,
  323,
  332,
  333,
  351,
  356,
  364,
  377,
  388,
  399,
  403,
  588,
  620,
  680,
  685,
  689,
  695,
  699,
  706,
  710,
  711,
  720,
  725,
  767,
  770,
  808,
  810,
  812,
  830,
  844,
  846,
  849,
  850,
  852,
  909,
  933,
  941,
  951,
  961,
  963,
  967,
  975,
  990,
  997,
  1004,
  1266,
  1282,
  1330,
  1605,
  1984,
  2010,
  2198,
  2266,
  2268,
  2284,
  2332,
  2334,
  2385,
  2392,
  2406,
  2408,
  2409,
  2422,
  2432,
  2453,
  2473,
  2491,
  2497,
  2505,
  2525,
  2528,
  2554,
  2575,
  2944,
  3834,
  3931,
  5307,
  5992,
  6035,
  6413,
  6917,
  6989,
  6990,
  7015,
  7034,
  7039,
  7110,
  7136,
  7418,
  7446,
  7497,
  7499,
  7576,
  7642,
  7661,
  7665,
  7667,
  7668,
  7675,
  7751,
  7756,
  7807,
  7819,
  7823,
  7895,
  7925,
  7926,
  7953

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
from tqdm import tqdm

BUCKET = "openaq-data-archive"
YEARS = range(2022, 2026)
OUT_DIR = "./openaq_raw2"
os.makedirs(OUT_DIR, exist_ok=True)
# Retry with exponential backoff to handle S3 throttling
s3 = boto3.client("s3", config=Config(
    signature_version=UNSIGNED,
    max_pool_connections=64,
    retries={"max_attempts": 10, "mode": "adaptive"},
))

def download_location_year(args):
    country, loc_id, year = args
    prefix = f"records/csv.gz/locationid={loc_id}/year={year}/"
    paginator = s3.get_paginator("list_objects_v2")

    downloaded = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            local_path = os.path.join(OUT_DIR, country, key)
            if os.path.exists(local_path):
                continue
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            s3.download_file(BUCKET, key, local_path)
            downloaded += 1
    return country, loc_id, year, downloaded

tasks = [
    (country, loc_id, year)
    for country, ids in location_ids.items()
    for loc_id in ids
    for year in YEARS
]

with ThreadPoolExecutor(max_workers=64) as executor:
    futures = {executor.submit(download_location_year, t): t for t in tasks}
    with tqdm(total=len(tasks), desc="Downloading", unit="task") as pbar:
        for future in as_completed(futures):
            country, loc_id, year, n = future.result()
            pbar.update(1)

Downloading: 100%|██████████| 704/704 [25:40<00:00,  2.19s/task]  
